<a href="https://colab.research.google.com/github/Kubojah-Dan/kuboja-codeboosters-2026/blob/main/DAY5/Day5_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Machine Learning (Supervised and UnSupervised Learning)

In [11]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [12]:
#==========================================
# Load The Student Dataset
#==========================================

df = pd.read_csv("student_performance.csv")

df.head()
df.info()
df.describe()
print(df.isnull().sum())
print(f"Number of Columns: {df.shape[1]}")
print(f"Number of Rows: {df.shape[0]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   student_id             30 non-null     int64 
 1   name                   30 non-null     object
 2   age                    30 non-null     int64 
 3   gender                 30 non-null     object
 4   department             30 non-null     object
 5   semester               30 non-null     int64 
 6   math_score             30 non-null     int64 
 7   science_score          30 non-null     int64 
 8   english_score          30 non-null     int64 
 9   programming_score      30 non-null     int64 
 10  attendance_percentage  30 non-null     int64 
 11  city                   30 non-null     object
 12  admission_year         30 non-null     int64 
dtypes: int64(9), object(4)
memory usage: 3.2+ KB
student_id               0
name                     0
age                      0
g

In [13]:
#======================================
# Understand the prediction task
#======================================

print("==== ML TASK DEFINITION ====")
print("TARGET : programming_score (the number we want to predict)")

==== ML TASK DEFINITION ====
TARGET : programming_score (the number we want to predict)


In [17]:
#================================================
# Feature Engineering: Encode categorical columns
#================================================

df_nl = df.copy()
le_gender = LabelEncoder()
df_nl["gender_encoded"] = le_gender.fit_transform(df_nl["gender"])
print(f"Gender encoding: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")
le_dept = LabelEncoder()
df_nl["dept_encoded"] = le_dept.fit_transform(df_nl["department"])
print(f"Department encoding: {dict(zip(le_dept.classes_, le_dept.transform(le_dept.classes_)))}")


print("\nNew columns added: gender_encoded, dept_encoded")
df_nl[['gender', 'gender_encoded', 'department', 'dept_encoded']].head(5)


Gender encoding: {'Female': np.int64(0), 'Male': np.int64(1)}
Department encoding: {'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}

New columns added: gender_encoded, dept_encoded


,gender,gender_encoded,department,dept_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [19]:
feature_cols = [
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'dept_encoded'
]

X = df_nl[feature_cols]
y = df_nl['programming_score']

print(f"Feature matrix X shape: {X.shape} (students x features)")
print(f"Target vector y shape: {y.shape} (one score per student)")
print(f"\nFeature columns: {feature_cols}")
print(f"Target range: {y.min()} to {y.max()}")

Feature matrix X shape: (30, 6) (students x features)
Target vector y shape: (30,) (one score per student)

Feature columns: ['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'dept_encoded']
Target range: 38 to 97


In [21]:
#================================
# Train/Test Split
#================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Total students: {len(X)}")
print(f"Training students: {(X_train)} {len(X_train)/len(X)*100:.0f}%")
print(f"\nTraining target range: {y_train.min()} to {y_train.max()}")
print(f"Testing target range: {y_test.min()} to {y_test.max()}")

Total students: 30
Training students:     math_score  science_score  ...  gender_encoded  dept_encoded
28          75             76  ...               1             3
24          86             82  ...               1             1
12          83             86  ...               1             1
0           85             78  ...               1             1
4           92             88  ...               1             1
16          56             61  ...               1             0
5           58             66  ...               0             2
13          74             78  ...               0             3
11          67             72  ...               0             0
22          61             66  ...               1             0
1           76             82  ...               0             1
2           65             74  ...               1             2
25          72             77  ...               0             2
3           70             80  ...               0  

In [22]:
#========================================
# Scale Features with StandardScaler
#========================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Feature matrix X_train_scaled shape: {X_train_scaled.shape}")
print(f"Feature matrix X_test_scaled shape: {X_test_scaled.shape}")
print(f"Training feature mean (should be ⏕ 0): {X_train_scaled.mean():.2f}")
print(f"Training feature std (should be ⏕ 1): {X_train_scaled.std():.2f}")

Feature matrix X_train_scaled shape: (24, 6)
Feature matrix X_test_scaled shape: (6, 6)
Training feature mean (should be ⏕ 0): -0.00
Training feature std (should be ⏕ 1): 1.00


In [23]:
from scipy.optimize import linear_sum_assignment
#========================================
# Train Model 1: Linear Regression
#========================================

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_mse = mean_squared_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print(f"Linear Regression MAE: {lr_mae:.2f} points on average")
print(f"Linear Regression MSE: {lr_mse:.2f} (error with penalty for large mistakes)")
print(f"Linear Regression R^2: {lr_r2:.2f}% of programming score variation explained")
print()
print(f"Linear Regression Coefficients (importance of each feature):")
for feature, coef in zip(feature_cols, lr_model.coef_):
    print(f"{feature}: {coef:.2f}")
print(f" {"bias (intercept)":<28}: {lr_model.intercept_:.2f}")

Linear Regression MAE: 9.37 points on average
Linear Regression MSE: 131.54 (error with penalty for large mistakes)
Linear Regression R^2: 0.74% of programming score variation explained

Linear Regression Coefficients (importance of each feature):
math_score: 20.26
science_score: 7.61
english_score: 2.40
attendance_percentage: -12.89
gender_encoded: -0.40
dept_encoded: -0.45
 bias (intercept)            : 68.04


In [25]:
#================================
# Decision Trees
#================================

dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

dt_mae = mean_absolute_error(y_test, dt_pred)
dt_mse = mean_squared_error(y_test, dt_pred)
dt_r2 = r2_score(y_test, dt_pred)

print("=== Model 2: Decision Tree (max_depth=5) ===")
print(f"MAE: {dt_mae:.2f} points on average")
print(f"MSE: {dt_mse:.2f} (error with penalty for large mistakes)")
print(f"R^2: {dt_r2:.2f}% of programming score variation explained")

=== Model 2: Decision Tree (max_depth=5) ===
MAE: 10.67 points on average
MSE: 231.67 (error with penalty for large mistakes)
R^2: 0.53% of programming score variation explained


In [26]:
#==================================
# Random Forest
#==================================

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("=== Model 3: Random Forest (100 trees) ===")
print(f"MAE: {rf_mae:.2f} points on average")
print(f"MSE: {rf_mse:.2f} (error with penalty for large mistakes)")
print(f"R^2: {rf_r2:.2f}% of programming score variation explained")

=== Model 3: Random Forest (100 trees) ===
MAE: 10.60 points on average
MSE: 195.04 (error with penalty for large mistakes)
R^2: 0.61% of programming score variation explained
